# RaBitQ Native IVF Plots

Plots average relative error, ADC time per pair, and relative-error-vs-time tradeoffs for the native `RaBitQ-Library` IVF sweep.

Inputs: `/mnthdd/cpanourg/2-hdvc/results/rabitq/*_RaBitQ-Library_adc_vs_exact_eval.csv`

Outputs: `/mnthdd/cpanourg/2-hdvc/results/rabitq/figures/RaBitQ_native_ivf`


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter

plt.rcParams.update({
    "font.size": 20,
    "axes.titlesize": 30,
    "axes.labelsize": 30,
    "xtick.labelsize": 24,
    "ytick.labelsize": 24,
    "legend.fontsize": 16,
})
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

PROJECT_ROOT = Path("/home/cpanourg/projects/2-hdvc")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = Path("/mnthdd/cpanourg/2-hdvc/results/rabitq")
FIGURES_DIR = DATA_DIR / "figures" / "RaBitQ_native_ivf"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

DATASET_ORDER = ["deep", "bigann", "gist", "msmarco", "openai"]
FIGURE_SAVE_FORMATS = ("pdf", "svg")
ADC_TIME_COL = "per_pair_s"
ADC_TIME_LABEL = "ADC time per pair (s)"
METHOD_LABEL = "RaBitQ-Library"
BITS_COL = "nbits"
CENTROIDS_COL = "n_centroids"

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)


In [ ]:
def save_figure(fig, output_dir: Path, stem: str):
    output_dir.mkdir(parents=True, exist_ok=True)
    saved = []
    for ext in FIGURE_SAVE_FORMATS:
        path = output_dir / f"{stem}.{ext}"
        fig.savefig(path, bbox_inches="tight")
        saved.append(path)
    print("Saved " + " and ".join(str(p) for p in saved))


def style_axes(ax, grid_axis="y"):
    ax.grid(True, axis=grid_axis, alpha=0.3)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="both", labelsize=24)


def set_sci_axes(ax, x=False, y=False):
    if x:
        fmt = ScalarFormatter(useMathText=True)
        fmt.set_powerlimits((-2, 3))
        ax.xaxis.set_major_formatter(fmt)
    if y:
        fmt = ScalarFormatter(useMathText=True)
        fmt.set_powerlimits((-2, 3))
        ax.yaxis.set_major_formatter(fmt)


def set_integer_xticks(ax, values, max_ticks: int = 6):
    vals = sorted({int(round(float(v))) for v in values if pd.notna(v)})
    if not vals:
        return
    if len(vals) > max_ticks:
        idx = np.linspace(0, len(vals) - 1, max_ticks).round().astype(int)
        vals = [vals[i] for i in sorted(set(idx))]
    ax.set_xticks(vals)
    ax.set_xticklabels([str(v) for v in vals], rotation=0)


def create_color_map(values, cmap_name="tab10"):
    vals = list(sorted(values))
    cmap = plt.get_cmap(cmap_name)
    return {v: cmap(i % cmap.N) for i, v in enumerate(vals)}


def dataset_short_name(name: str) -> str:
    mapping = {
        "deep10k": "deep",
        "bigann10k": "bigann",
        "gist10k": "gist",
        "msmarco10k": "msmarco",
        "openai10k": "openai",
    }
    return mapping.get(str(name), str(name))


In [ ]:
def load_rabitq_native_ivf_data(data_dir: Path = DATA_DIR) -> pd.DataFrame:
    csv_paths = sorted(data_dir.glob("*_RaBitQ-Library_adc_vs_exact_eval.csv"))
    if not csv_paths:
        raise FileNotFoundError(f"No native RaBitQ CSV files found in {data_dir}")

    frames = []
    for path in csv_paths:
        df = pd.read_csv(path)
        if "dataset" not in df.columns:
            df["dataset"] = path.name.split("_")[0]
        frames.append(df)

    df = pd.concat(frames, ignore_index=True)
    numeric_cols = [
        "nbits", "bits_per_vector", "train_size", "train_time_s", "encoding_time_s",
        "distance_table_time_s", "cdist_time_s", "adc_time_s", "adc_time_s_std",
        "per_query_us_mean", "per_pair_ns_mean", "rel_error_mean", "rel_error_std",
        "nq", "nb", "nb_sample", "nq_sample", "dim", "n_subquantizers",
        "seed", "n_centroids", "nprobe",
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df["dataset"] = df["dataset"].map(dataset_short_name)
    df[ADC_TIME_COL] = df["per_pair_ns_mean"] * 1e-9
    df["per_pair_us"] = df["per_pair_ns_mean"] * 1e-3
    df["config_label"] = df.apply(lambda r: f"K={int(r[CENTROIDS_COL])}, b={int(r[BITS_COL])}", axis=1)
    df = df.sort_values(["dataset", CENTROIDS_COL, BITS_COL]).reset_index(drop=True)
    return df


plot_df = load_rabitq_native_ivf_data()
display(plot_df.head())
display(plot_df.groupby("dataset")[[BITS_COL, CENTROIDS_COL]].agg(["min", "max", "nunique"]))


In [ ]:
def aggregate_metric_df(df: pd.DataFrame, x_col: str, y_col: str, average_over: str) -> pd.DataFrame:
    keep = ["dataset", x_col, average_over, y_col]
    agg = (
        df[keep]
        .dropna(subset=[x_col, y_col, average_over])
        .groupby(["dataset", x_col], as_index=False)
        .agg({y_col: ["mean", "std", "count"]})
    )
    agg.columns = ["dataset", x_col, f"{y_col}_mean", f"{y_col}_std", "count"]
    return agg.sort_values(["dataset", x_col]).reset_index(drop=True)


def plot_avg_metric_vs_param(
    df: pd.DataFrame,
    x_col: str,
    y_col: str,
    x_label: str,
    y_label: str,
    average_over: str,
    output_stem: str,
    datasets=DATASET_ORDER,
    sci_y: bool = False,
):
    agg = aggregate_metric_df(df, x_col=x_col, y_col=y_col, average_over=average_over)
    color = "tab:blue"
    for dataset in datasets:
        sub = agg[agg["dataset"].eq(dataset)].dropna(subset=[x_col, f"{y_col}_mean"])
        if sub.empty:
            continue
        fig, ax = plt.subplots()
        ax.plot(sub[x_col], sub[f"{y_col}_mean"], marker="o", color=color, linewidth=2.5, markersize=12)
        if sub[f"{y_col}_std"].notna().any():
            yerr = sub[f"{y_col}_std"].fillna(0.0)
            ax.fill_between(sub[x_col], sub[f"{y_col}_mean"] - yerr, sub[f"{y_col}_mean"] + yerr, color=color, alpha=0.15)
        ax.set_xlabel(x_label)
        ax.set_ylabel(y_label)
        ax.set_title(f"{dataset}: avg over {average_over}")
        style_axes(ax, grid_axis="y")
        if x_col in {BITS_COL, CENTROIDS_COL}:
            set_integer_xticks(ax, sub[x_col])
        set_sci_axes(ax, y=sci_y)
        save_figure(fig, FIGURES_DIR / dataset, f"{dataset}_{output_stem}")
        plt.show()
        plt.close(fig)


In [ ]:
def plot_metric_vs_param_with_config_legend(
    df: pd.DataFrame,
    x_col: str,
    y_col: str,
    x_label: str,
    y_label: str,
    legend_col: str,
    output_stem: str,
    datasets=DATASET_ORDER,
    sci_y: bool = False,
):
    for dataset in datasets:
        sub = df[df["dataset"].eq(dataset)].dropna(subset=[x_col, y_col, legend_col]).copy()
        if sub.empty:
            continue
        color_map = create_color_map(sub[legend_col].unique())
        fig, ax = plt.subplots(figsize=(11, 7))
        for value, curve in sub.groupby(legend_col, sort=True):
            curve = curve.sort_values(x_col)
            ax.plot(
                curve[x_col], curve[y_col], marker="o", linewidth=2.2, markersize=10,
                color=color_map[value], label=f"{legend_col}={int(value)}"
            )
        ax.set_xlabel(x_label)
        ax.set_ylabel(y_label)
        ax.set_title(dataset)
        style_axes(ax, grid_axis="y")
        if x_col in {BITS_COL, CENTROIDS_COL}:
            set_integer_xticks(ax, sub[x_col])
        set_sci_axes(ax, y=sci_y)
        ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False)
        save_figure(fig, FIGURES_DIR / dataset, f"{dataset}_{output_stem}")
        plt.show()
        plt.close(fig)


In [ ]:
def pareto_frontier_minimize(df: pd.DataFrame, x_col: str, y_col: str) -> pd.DataFrame:
    ordered = df.sort_values([x_col, y_col]).reset_index(drop=True)
    keep_rows = []
    best_y = np.inf
    for _, row in ordered.iterrows():
        y = row[y_col]
        if y <= best_y:
            keep_rows.append(True)
            best_y = y
        else:
            keep_rows.append(False)
    return ordered.loc[keep_rows].reset_index(drop=True)


def plot_relerr_vs_adc_pareto(
    df: pd.DataFrame,
    output_stem: str,
    datasets=DATASET_ORDER,
    annotate: bool = True,
):
    for dataset in datasets:
        sub = df[df["dataset"].eq(dataset)].dropna(subset=[ADC_TIME_COL, "rel_error_mean"]).copy()
        if sub.empty:
            continue
        fig, ax = plt.subplots(figsize=(11, 7))
        color_map = create_color_map(sub[CENTROIDS_COL].unique())
        for _, row in sub.iterrows():
            ax.scatter(row[ADC_TIME_COL], row["rel_error_mean"], color=color_map[row[CENTROIDS_COL]], s=90, alpha=0.9)
            if annotate:
                ax.annotate(f"K={int(row[CENTROIDS_COL])}, b={int(row[BITS_COL])}", (row[ADC_TIME_COL], row["rel_error_mean"]), textcoords="offset points", xytext=(6, 4), fontsize=11)
        frontier = pareto_frontier_minimize(sub[[ADC_TIME_COL, "rel_error_mean"]].copy(), ADC_TIME_COL, "rel_error_mean")
        ax.plot(frontier[ADC_TIME_COL], frontier["rel_error_mean"], color="black", linewidth=1.8, alpha=0.8)
        ax.scatter(frontier[ADC_TIME_COL], frontier["rel_error_mean"], color="black", s=35, alpha=0.8)
        ax.set_xlabel(ADC_TIME_LABEL)
        ax.set_ylabel("Average Relative Error")
        ax.set_title(f"{dataset}: relerr vs ADC time per pair")
        style_axes(ax, grid_axis="both")
        set_sci_axes(ax, x=True, y=False)
        handles = [plt.Line2D([0], [0], marker="o", color="w", label=f"K={int(k)}", markerfacecolor=c, markersize=10) for k, c in color_map.items()]
        ax.legend(handles=handles, loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False)
        save_figure(fig, FIGURES_DIR / dataset, f"{dataset}_{output_stem}")
        plt.show()
        plt.close(fig)


## Relative Error

Average relative error versus bitwidth and IVF centroid count, plus per-configuration curves.

In [ ]:
plot_avg_metric_vs_param(
    plot_df,
    x_col=BITS_COL,
    y_col="rel_error_mean",
    x_label="Bits per dimension",
    y_label="Average Relative Error",
    average_over=CENTROIDS_COL,
    output_stem="avg_relerr_vs_nbits",
)

plot_avg_metric_vs_param(
    plot_df,
    x_col=CENTROIDS_COL,
    y_col="rel_error_mean",
    x_label="IVF centroids",
    y_label="Average Relative Error",
    average_over=BITS_COL,
    output_stem="avg_relerr_vs_ncentroids",
)

plot_metric_vs_param_with_config_legend(
    plot_df,
    x_col=BITS_COL,
    y_col="rel_error_mean",
    x_label="Bits per dimension",
    y_label="Average Relative Error",
    legend_col=CENTROIDS_COL,
    output_stem="relerr_vs_nbits_by_centroids",
)

plot_metric_vs_param_with_config_legend(
    plot_df,
    x_col=CENTROIDS_COL,
    y_col="rel_error_mean",
    x_label="IVF centroids",
    y_label="Average Relative Error",
    legend_col=BITS_COL,
    output_stem="relerr_vs_ncentroids_by_bits",
)


## ADC Time Per Pair

Uses `per_pair_ns_mean` from the native sweep, converted to seconds per pair.

In [ ]:
plot_avg_metric_vs_param(
    plot_df,
    x_col=BITS_COL,
    y_col=ADC_TIME_COL,
    x_label="Bits per dimension",
    y_label=ADC_TIME_LABEL,
    average_over=CENTROIDS_COL,
    output_stem="avg_adc_time_per_pair_vs_nbits",
    sci_y=True,
)

plot_avg_metric_vs_param(
    plot_df,
    x_col=CENTROIDS_COL,
    y_col=ADC_TIME_COL,
    x_label="IVF centroids",
    y_label=ADC_TIME_LABEL,
    average_over=BITS_COL,
    output_stem="avg_adc_time_per_pair_vs_ncentroids",
    sci_y=True,
)

plot_metric_vs_param_with_config_legend(
    plot_df,
    x_col=BITS_COL,
    y_col=ADC_TIME_COL,
    x_label="Bits per dimension",
    y_label=ADC_TIME_LABEL,
    legend_col=CENTROIDS_COL,
    output_stem="adc_time_per_pair_vs_nbits_by_centroids",
    sci_y=True,
)

plot_metric_vs_param_with_config_legend(
    plot_df,
    x_col=CENTROIDS_COL,
    y_col=ADC_TIME_COL,
    x_label="IVF centroids",
    y_label=ADC_TIME_LABEL,
    legend_col=BITS_COL,
    output_stem="adc_time_per_pair_vs_ncentroids_by_bits",
    sci_y=True,
)


## Relative Error vs ADC Time

Scatter and Pareto-style plots over all `(K, bits)` configurations per dataset.

In [ ]:
plot_relerr_vs_adc_pareto(
    plot_df,
    output_stem="relerr_vs_adc_time_per_pair_pareto",
    annotate=True,
)
